In [ ]:
import io
import contextlib

# Global list to store captured output sections from different cells
_combined_report_sections = []

In [ ]:
# Install necessary libraries
!pip install requests pandas

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from google.colab import userdata

# Retrieve API token from Colab secrets
TOGGL_API_TOKEN = userdata.get('TOGGL_API_TOKEN')

if not TOGGL_API_TOKEN:
    raise ValueError("Toggl Track API token not found. Please add it to Colab secrets as 'TOGGL_API_TOKEN'.")

# Toggl Track API base URL
TOGGL_API_BASE_URL = 'https://api.track.toggl.com/api/v9'


In [ ]:
import pytz
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, HTML

# Define global variables to hold the selected dates/times
# Initialize them with default values so subsequent cells don't fail if button isn't clicked
global start_time_jst, end_time_jst, start_time_str, end_time_str, params

# Define JST timezone
jst = pytz.timezone('Asia/Tokyo')

# Get current UTC time and convert to JST for default values
now_utc = datetime.now(pytz.utc)
now_jst = now_utc.astimezone(jst)

print("--- 期間設定 ---")

# --- Calculate default end_time_jst (Last Sunday at 23:59:59 JST) ---
# 今日の曜日 (0=月, 6=日) に基づいて、直前の日曜日を特定
days_ago_to_last_sunday = (now_jst.weekday() + 1) % 7 # 0 for Sunday, 1 for Monday, etc.
last_sunday_date_jst_naive = now_jst.date() - timedelta(days=days_ago_to_last_sunday)
end_date_default = last_sunday_date_jst_naive
end_hour_default = 23
end_minute_default = 59

# --- Calculate default start_time_jst (Reporting week Monday at 00:00:00 JST) ---
# レポート対象週の開始日を計算 (end_date_default (直前の日曜日) の6日前)
start_date_default = end_date_default - timedelta(days=6)
start_hour_default = 0
start_minute_default = 0

# Create widgets for date and time input
start_date_picker = widgets.DatePicker(
    description='開始日:',
    value=start_date_default,
    disabled=False
)
start_hour_picker = widgets.BoundedIntText(
    description='時間:',
    min=0, max=23, step=1,
    value=start_hour_default,
    disabled=False
)
start_minute_picker = widgets.BoundedIntText(
    description='分:',
    min=0, max=59, step=1,
    value=start_minute_default,
    disabled=False
)

end_date_picker = widgets.DatePicker(
    description='終了日:',
    value=end_date_default,
    disabled=False
)
end_hour_picker = widgets.BoundedIntText(
    description='時間:',
    min=0, max=23, step=1,
    value=end_hour_default,
    disabled=False
)
end_minute_picker = widgets.BoundedIntText(
    description='分:',
    min=0, max=59, step=1,
    value=end_minute_default,
    disabled=False
)

# Arrange widgets for better display
start_widgets = widgets.VBox([
    widgets.HTML("<b>開始日時 (JST):</b>"),
    start_date_picker,
    widgets.HBox([start_hour_picker, start_minute_picker])
])

end_widgets = widgets.VBox([
    widgets.HTML("<b>終了日時 (JST):</b>"),
    end_date_picker,
    widgets.HBox([end_hour_picker, end_minute_picker])
])

display(widgets.HBox([start_widgets, end_widgets]))

# Create a button to confirm selections
confirm_button = widgets.Button(description="選択範囲を確定", button_style='success')
output_area = widgets.Output() # To display output without re-rendering widgets
display(confirm_button, output_area)

def on_confirm_button_clicked(b):
    with output_area:
        output_area.clear_output() # Clear previous output
        print("選択範囲を確定中...")

        # Declare global variables to be modified
        global start_time_jst, end_time_jst, start_time_str, end_time_str, params

        # Use widget values to define the actual start_time_jst and end_time_jst
        selected_start_date = start_date_picker.value
        selected_start_hour = start_hour_picker.value
        selected_start_minute = start_minute_picker.value

        selected_end_date = end_date_picker.value
        selected_end_hour = end_hour_picker.value
        selected_end_minute = end_minute_picker.value

        # Create naive datetime objects from widget values
        start_time_jst_naive = datetime(
            selected_start_date.year, selected_start_date.month, selected_start_date.day,
            selected_start_hour, selected_start_minute, 0
        )
        end_time_jst_naive = datetime(
            selected_end_date.year, selected_end_date.month, selected_end_date.day,
            selected_end_hour, selected_end_minute, 0
        )

        # Localize the naive datetime to JST
        start_time_jst = jst.localize(start_time_jst_naive)
        end_time_jst = jst.localize(end_time_jst_naive)

        # Convert to UTC for the API request
        start_time_utc = start_time_jst.astimezone(pytz.utc)
        end_time_utc = end_time_jst.astimezone(pytz.utc)

        # Print the calculated dates as requested by the user
        print(f"設定された開始日 (JST): {start_time_jst.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"設定された終了日 (JST): {end_time_jst.strftime('%Y-%m-%d %H:%M:%S')}")

        # Format times for API request (ISO 8601 in UTC)
        start_time_str = start_time_utc.isoformat().replace('+00:00', 'Z')
        end_time_str = end_time_utc.isoformat().replace('+00:00', 'Z')

        # Update params for the API request
        params = {
            'start_date': start_time_str,
            'end_date': end_time_str
        }

        print(f"\nFetching time entries from {start_time_jst} to {end_time_jst} (JST)...")
        print("これで次以降のセルを実行できます。")

# Attach the handler to the button
confirm_button.on_click(on_confirm_button_clicked)

# --- Initial setup of global variables with default values --- #
# This ensures that if the user runs subsequent cells WITHOUT clicking the button,
# they still have some valid (default) values.
start_time_jst_initial_naive = datetime(
    start_date_default.year, start_date_default.month, start_date_default.day,
    start_hour_default, start_minute_default, 0
)
end_time_jst_initial_naive = datetime(
    end_date_default.year, end_date_default.month, end_date_default.day,
    end_hour_default, end_minute_default, 0
)

start_time_jst = jst.localize(start_time_jst_initial_naive)
end_time_jst = jst.localize(end_time_jst_initial_naive)

start_time_utc_initial = start_time_jst.astimezone(pytz.utc)
end_time_utc_initial = end_time_jst.astimezone(pytz.utc)

start_time_str = start_time_utc_initial.isoformat().replace('+00:00', 'Z')
end_time_str = end_time_utc_initial.isoformat().replace('+00:00', 'Z')

# Toggl Track API endpoint for time entries (still needed for params)
time_entries_url = f"{TOGGL_API_BASE_URL}/me/time_entries" # TOGGL_API_BASE_URL is global from 9NXHq8wEICKJ

# Headers for authentication (still needed for params)
headers = {
    'Content-Type': 'application/json'
}

# Basic authentication using API token as password (still needed for params)
auth = (TOGGL_API_TOKEN, 'api_token') # TOGGL_API_TOKEN is global from 9NXHq8wEICKJ

params = {
    'start_date': start_time_str,
    'end_date': end_time_str
}
print(f"初期設定された期間 (JST): {start_time_jst.strftime('%Y-%m-%d %H:%M:%S')} - {end_time_jst.strftime('%Y-%m-%d %H:%M:%S')}")

--- 期間設定 ---


Button(button_style='success', description='選択範囲を確定', style=ButtonStyle())

Output()

初期設定された期間 (JST): 2026-08-31 00:00:00 - 2026-09-06 23:59:00


In [ ]:
import pandas as pd
from google.colab import files

# ワークスペースIDからラベルへのマッピングを作成
workspace_id_to_label = {v: k for k, v in workspaces_to_fetch.items()}

csv_rows = []
for entry in processed_time_entries:
    # ワークスペースID, 名前
    workspace_id = entry.get('workspace_id')
    workspace_name = workspace_id_to_label.get(workspace_id, 'Unknown Workspace')

    # プロジェクトID, 名前
    project_id = entry.get('project_id')
    project_name = project_id_to_name.get(project_id, 'No Project')

    # クライアントID, 名前 (プロジェクトIDからクライアントIDを検索)
    client_id = project_id_to_client_id.get(project_id)
    client_name = client_id_to_name.get(client_id, 'No Client') if client_id else 'No Client'

    # タスク名 (descriptionを使用)
    task_name = entry.get('description', '')

    # 開始日時と終了日時 (JSTでフォーマット)
    start_time_formatted = entry['start_jst'].strftime('%Y-%m-%d %H:%M:%S') if entry['start_jst'] else ''
    stop_time_formatted = entry['stop_jst'].strftime('%Y-%m-%d %H:%M:%S') if entry['stop_jst'] else ''

    # 経過時間 (format_duration関数を使用)
    duration_formatted = format_duration(entry.get('duration'))

    # タグ (コロン区切り)
    tags = entry.get('tags', [])
    tags_str = ":".join(tags) if tags else ''

    csv_rows.append({
        'ワークスペースID': workspace_id,
        'ワークスペース名': workspace_name,
        'プロジェクトID': project_id,
        'プロジェクト名': project_name,
        'クライアントID': client_id,
        'クライアント名': client_name,
        'タスク名': task_name,
        '開始日時': start_time_formatted,
        '終了日時': stop_time_formatted,
        '経過時間_秒': entry.get('duration'), # 秒単位での経過時間を保存
        '経過時間': duration_formatted,
        'タグ': tags_str
    })

# DataFrameを作成
df_activity_log = pd.DataFrame(csv_rows)

# CSVファイルとして保存
# report_date_jst は前方のセルで定義済みと仮定
output_filename = f"toggl_activity_log_full_{report_date_jst.strftime('%Y-%m-%d')}.csv"
df_activity_log.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"行動ログが '{output_filename}' としてCSV形式で保存されました。")
display(df_activity_log.head())
files.download(output_filename)

NameError: name 'processed_time_entries' is not defined

In [ ]:
try:
    response = requests.get(time_entries_url, headers=headers, auth=auth, params=params)
    response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)
    time_entries_data = response.json()
    print(f"Successfully fetched {len(time_entries_data)} time entries.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching data: {e}")
    if response.status_code == 403:
        print("Check your API token and ensure it's correct.")
    time_entries_data = [] # Ensure data is empty if an error occurs

In [ ]:
import requests
from google.colab import userdata

# Define the two workspaces with their IDs and labels
# IDs are now fetched from Colab secrets
work_workspace_id = userdata.get('TOGGL_WORK_WORKSPACE_ID')
life_workspace_id = userdata.get('TOGGL_LIFE_WORKSPACE_ID')

if not work_workspace_id or not life_workspace_id:
    raise ValueError("Toggl workspace IDs not found. Please add 'TOGGL_WORK_WORKSPACE_ID' and 'TOGGL_LIFE_WORKSPACE_ID' to Colab secrets.")

workspaces_to_fetch = {
    'work': int(work_workspace_id),
    'life': int(life_workspace_id)
}

combined_workspace_data = {
    'projects': [],
    'clients': []
}

print("Fetching project and client data for specified workspaces...")

for label, workspace_id in workspaces_to_fetch.items():
    print(f"\n--- Processing Workspace: {label} (ID: {workspace_id}) ---")
    try:
        # --- Fetch Projects Data for current workspace ---
        all_projects_for_workspace = []

        # Fetch active projects (default behavior, or explicitly active=true)
        projects_url_active = f"{TOGGL_API_BASE_URL}/workspaces/{workspace_id}/projects?active=true"
        print(f"  Fetching active projects from: {projects_url_active}")
        projects_response_active = requests.get(projects_url_active, headers=headers, auth=auth)
        projects_response_active.raise_for_status()
        active_projects_data = projects_response_active.json()
        print(f"  Successfully fetched {len(active_projects_data)} active projects for {label}.")
        all_projects_for_workspace.extend(active_projects_data)

        # Fetch archived projects (active=false)
        projects_url_archived = f"{TOGGL_API_BASE_URL}/workspaces/{workspace_id}/projects?active=false"
        print(f"  Fetching archived projects from: {projects_url_archived}")
        projects_response_archived = requests.get(projects_url_archived, headers=headers, auth=auth)
        projects_response_archived.raise_for_status()
        archived_projects_data = projects_response_archived.json()
        print(f"  Successfully fetched {len(archived_projects_data)} archived projects for {label}.")
        all_projects_for_workspace.extend(archived_projects_data)

        # Add workspace label to each project and append to combined list
        for project in all_projects_for_workspace:
            project['workspace_label'] = label
            combined_workspace_data['projects'].append(project)

        print(f"  Total projects fetched for {label}: {len(all_projects_for_workspace)}")

        # --- Fetch Clients Data for current workspace ---
        clients_url = f"{TOGGL_API_BASE_URL}/workspaces/{workspace_id}/clients"
        print(f"  Fetching clients from: {clients_url}")
        clients_response = requests.get(clients_url, headers=headers, auth=auth)
        clients_response.raise_for_status()
        clients_data = clients_response.json()
        print(f"  Successfully fetched {len(clients_data)} clients for {label}.")

        # Add workspace label to each client and append to combined list
        for client in clients_data:
            client['workspace_label'] = label
            combined_workspace_data['clients'].append(client)

    except requests.exceptions.RequestException as e:
        print(f"  Error fetching data for workspace {label}: {e}")
        status_code = None
        if 'projects_response_active' in locals() and hasattr(projects_response_active, 'status_code'):
            status_code = projects_response_active.status_code
        elif 'projects_response_archived' in locals() and hasattr(projects_response_archived, 'status_code'):
            status_code = projects_response_archived.status_code
        elif 'clients_response' in locals() and hasattr(clients_response, 'status_code'):
            status_code = clients_response.status_code

        if status_code == 403:
            print(f"  Check your API token and ensure it's correct and has access to workspace {workspace_id}.")

print("\n--- Combined Project List ---")
if combined_workspace_data['projects']:
    for project in combined_workspace_data['projects']:
        print(f"[Workspace: {project['workspace_label']}] Project ID: {project['id']}, Name: {project['name']}, Client ID: {project.get('client_id', 'N/A')}")
else:
    print("No projects found in any workspace.")

print("\n--- Combined Client List ---")
if combined_workspace_data['clients']:
    for client in combined_workspace_data['clients']:
        print(f"[Workspace: {client['workspace_label']}] Client ID: {client['id']}, Name: {client['name']}")
else:
    print("No clients found in any workspace.")

### レポート対象期間の計算とタイムエントリのフィルタリング

Toggl Trackのタイムエントリを週次レポート用に準備するため、まず「レポート対象週」（先週の月曜日から直前の日曜日まで）と「比較対象週」（その前の週）の日付範囲を正確に計算します。次に、既存のタイムエントリデータ (`time_entries_data`) をこれらの期間でフィルタリングし、それぞれの週のデータを抽出します。

In [ ]:
import pytz
from datetime import datetime, timedelta
import pandas as pd
from google.colab import drive

# Mount Google Drive (will skip if already mounted)
try:
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print(f"Google Drive already mounted or error mounting: {e}")

jst = pytz.timezone('Asia/Tokyo') # JST timezone is already defined in 034f6343

# Date variables (start_time_report_week_jst, end_time_report_week_jst, report_date_jst)
# are already defined in the previous cell (034f6343) and are correct.

print(f"レポート対象期間 (JST): {start_time_report_week_jst.strftime('%Y-%m-%d %H:%M:%S')} - {end_time_report_week_jst.strftime('%Y-%m-%d %H:%M:%S')}")

# --- 比較対象週の期間計算 (レポート対象週の前の週) ---
# 比較対象週の終了日 (レポート対象週の開始日の前日 23:59:59 JST)
end_time_comparison_week_jst = start_time_report_week_jst - timedelta(seconds=1)
# 比較対象週の開始日 (レポート対象週の開始日から1週間前) # 修正箇所
start_time_comparison_week_jst = start_time_report_week_jst - timedelta(weeks=1)

print(f"比較対象期間 (JST): {start_time_comparison_week_jst.strftime('%Y-%m-%d %H:%M:%S')} - {end_time_comparison_week_jst.strftime('%Y-%m-%d %H:%M:%S')}")

# --- Helper function to convert H:MM:SS to seconds ---
def parse_duration_to_seconds(duration_str):
    if pd.isna(duration_str) or duration_str == 'N/A': # Check for NaN or 'N/A'
        return 0

    # Try to parse as float first for '経過時間_秒' if it was saved as number type but read as string
    try:
        return float(duration_str)
    except ValueError:
        pass # Fallback to H:MM:SS parsing

    # Ensure it's a string before splitting
    duration_str = str(duration_str)

    parts = list(map(int, duration_str.split(':')))
    if len(parts) == 3:
        h, m, s = parts
    elif len(parts) == 2: # MM:SS
        h = 0
        m, s = parts
    elif len(parts) == 1: # SS
        h = 0
        m = 0
        s = parts[0]
    else:
        return 0 # Invalid format
    return h * 3600 + m * 60 + s

# --- Load previous week's data from Google Drive for comparison ---
time_entries_comparison_week = []
previous_week_filename_date_str = end_time_comparison_week_jst.strftime('%Y-%m-%d')
gdrive_prev_week_path = f'/content/drive/MyDrive/TogglTrack_Reports/toggl_activity_log_full_{previous_week_filename_date_str}.csv'

print(f"比較対象週のデータをGoogle Driveから読み込み中: {gdrive_prev_week_path}")
try:
    df_comparison_log = pd.read_csv(gdrive_prev_week_path)
    print(f"'{gdrive_prev_week_path}' から {len(df_comparison_log)} 件の比較対象週データを読み込みました。")

    # Inverse mapping from workspace label ('work', 'life') to its ID (integer)
    # workspaces_to_fetch is defined in dcfd22df
    workspace_label_to_id = {label: wid for label, wid in workspaces_to_fetch.items()}

    # NEW: Create inverse mappings for project and client names to their IDs
    # project_id_to_name is defined in fb1cfdc7, client_name_to_id is defined in hgBizH8JjLJS
    project_name_to_id = {name: pid for pid, name in project_id_to_name.items()}

    # Convert DataFrame rows back into the time_entries_data format (list of dicts)
    processed_time_entries_comparison_temp = [] # 一時リストを作成
    for index, row in df_comparison_log.iterrows():
        entry = {}
        # Get workspace_id by mapping the 'ワークスペース名' from CSV to the actual ID
        workspace_name_from_csv = row.get('ワークスペース名')
        entry['workspace_id'] = workspace_label_to_id.get(workspace_name_from_csv)

        # Get project_id by mapping the 'プロジェクト名' from CSV to the actual ID
        project_name_from_csv = row.get('プロジェクト名')
        entry['project_id'] = project_name_to_id.get(project_name_from_csv)

        # Get client_id by mapping the 'クライアント名' from CSV to the actual ID
        client_name_from_csv = row.get('クライアント名')
        entry['client_id'] = client_name_to_id.get(client_name_from_csv) # Using client_name_to_id

        entry['description'] = row.get('タスク名')

        # '経過時間_秒' があればそれを使用、なければ '経過時間' (H:MM:SS) を解析
        entry['duration'] = row.get('経過時間_秒', parse_duration_to_seconds(row.get('経過時間')))

        # Convert '開始日時' and '終了日時' to JST datetime objects
        start_time_str = row.get('開始日時')
        stop_time_str = row.get('終了日時')
        entry['start_jst'] = jst.localize(datetime.strptime(str(start_time_str), '%Y-%m-%d %H:%M:%S')) if pd.notna(start_time_str) else None
        entry['stop_jst'] = jst.localize(datetime.strptime(str(stop_time_str), '%Y-%m-%d %H:%M:%S')) if pd.notna(stop_time_str) else None

        # Add tags (assuming they were saved as colon-separated string)
        tags_str = row.get('タグ', '')
        entry['tags'] = str(tags_str).split(':') if pd.notna(tags_str) and tags_str else []

        processed_time_entries_comparison_temp.append(entry)

    # 読み込んだデータを比較対象週の期間でフィルタリング
    time_entries_comparison_week = [
        entry for entry in processed_time_entries_comparison_temp
        if entry['start_jst'] and start_time_comparison_week_jst <= entry['start_jst'] <= end_time_comparison_week_jst
    ]

except FileNotFoundError:
    print(f"比較対象週のデータファイル '{gdrive_prev_week_path}' が見つかりませんでした。比較対象週のデータは空として処理されます。")
except Exception as e:
    print(f"比較対象週のデータ読み込み中にエラーが発生しました: {e}。比較対象週のデータは空として処理されます。")
    time_entries_comparison_week = [] # Ensure it's empty on error

# time_entries_report_week is already processed and filtered in cell 034f6343

print(f"レポート対象週のタイムエントリ数: {len(time_entries_report_week)}")
print(f"比較対象週のタイムエントリ数 (ファイルから読み込み、フィルタリング後): {len(time_entries_comparison_week)}")

### ヘルパー関数の定義とデータの集計

時間を`H:MM:SS`形式に整形するためのヘルパー関数と、タイムエントリリストを集計して総作業時間、日平均、プロジェクトごとの時間を算出する関数を定義します。これらの関数を用いて、レポート対象週と比較対象週のデータを処理します。

In [ ]:
from datetime import timedelta

# Toggl TrackのプロジェクトIDとプロジェクト名をマッピングするための辞書を作成
# combined_workspace_data は前のセルで取得済みと仮定
project_id_to_name = {}
project_id_to_client_id = {}
for project in combined_workspace_data['projects']:
    project_id_to_name[project['id']] = project['name']
    if project.get('client_id'): # クライアントが紐付いている場合のみ追加
        project_id_to_client_id[project['id']] = project['client_id']

# クライアントIDからクライアント名へのマッピングを作成
client_id_to_name = {}
for client in combined_workspace_data['clients']:
    client_id_to_name[client['id']] = client['name']

# ワークスペースIDからラベルへのマッピングを作成 (51pss7TwWZVi で使用)
workspace_id_to_label = {v: k for k, v in workspaces_to_fetch.items()}

# 時間をH:MM:SS形式にフォーマットするヘルパー関数
def format_duration(seconds):
    if seconds is None:
        return "N/A"
    seconds = int(seconds)
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    return f"{hours:02}:{minutes:02}:{secs:02}"

# 時間差分を+H:MM:SSまたは-H:MM:SS形式にフォーマットするヘルパー関数
def format_duration_diff(seconds_diff):
    if seconds_diff is None:
        return ""
    sign = "+" if seconds_diff >= 0 else "-"
    # 小数点以下を切り捨てるため、abs後にintにキャスト
    abs_seconds_diff = int(abs(seconds_diff))
    hours = abs_seconds_diff // 3600
    minutes = (abs_seconds_diff % 3600) // 60
    secs = abs_seconds_diff % 60
    return f"{sign}{hours:02}:{minutes:02}:{secs:02}"

# 期間内のタイムエントリを処理し、集計する関数
def process_time_entries_for_period(entries_list, project_name_lookup):
    total_duration_seconds = 0
    project_durations = {} # キー: project_name, 値: 合計時間 (秒)

    for entry in entries_list:
        duration = entry.get('duration', 0) # durationは秒単位
        total_duration_seconds += duration

        project_id = entry.get('project_id')
        # プロジェクトIDがNoneの場合、またはproject_id_to_nameに存在しない場合は'No Project'と表示
        project_name = project_name_lookup.get(project_id, 'No Project')

        project_durations[project_name] = project_durations.get(project_name, 0) + duration

    # 1週間のレポートなので、日平均は7で割る
    daily_average_seconds = total_duration_seconds / 7 if total_duration_seconds > 0 else 0

    return total_duration_seconds, daily_average_seconds, project_durations

# 'work' ワークスペースのIDを取得
WORK_WORKSPACE_ID = workspaces_to_fetch.get('work')

# レポート対象週のタイムエントリを'work'ワークスペースでフィルタリング
filtered_time_entries_report_week = [
    entry for entry in time_entries_report_week if entry.get('workspace_id') == WORK_WORKSPACE_ID
]

# 比較対象週のタイムエントリを'work'ワークスペースでフィルタリング
filtered_time_entries_comparison_week = [
    entry for entry in time_entries_comparison_week if entry.get('workspace_id') == WORK_WORKSPACE_ID
]

# フィルタリングされたデータを使用してレポート対象週のデータを処理
total_duration_report, daily_avg_report, project_durations_report = process_time_entries_for_period(
    filtered_time_entries_report_week, project_id_to_name
)

# フィルタリングされたデータを使用して比較対象週のデータを処理
total_duration_comparison, daily_avg_comparison, project_durations_comparison = process_time_entries_for_period(
    filtered_time_entries_comparison_week, project_id_to_name
)

# 各種集計を出力する。
各種集計結果をMarkdown形式で出力する処理群

In [ ]:
import sys # <-- Add this import
import io
import contextlib

# Create a temporary buffer for current cell's output
temp_buf = io.StringIO()

with contextlib.redirect_stdout(temp_buf): # Redirect all prints within this block to temp_buf
    # Add a check for comparison data
    if total_duration_comparison == 0 and total_duration_report > 0:
        print("--- 注意 ---")
        print("比較対象週のデータが検出されなかったか、またはフィルタリングの結果ゼロになりました。")
        print("これにより、比較差分は現在の週の合計時間と同じ値として表示されています。")
        print("比較を正しく行うためには、前週のデータファイルが正しく生成されていることを確認してください。")
        print("------------\n")

    # 差分を計算
    total_duration_diff = total_duration_report - total_duration_comparison
    daily_avg_diff = daily_avg_report - daily_avg_comparison

    print(f"{start_time_report_week_jst.strftime('%Y年%m月%d日')}から{end_time_report_week_jst.strftime('%Y年%m月%d日')}までのレポート")
    print(f"* 計算日: {report_date_jst.strftime('%Y/%m/%d')}")
    print(f"* ワークスペースの総作業時間: {format_duration(total_duration_report)}({format_duration_diff(total_duration_diff)})")
    print(f"* 平均: {format_duration(daily_avg_report)}({format_duration_diff(daily_avg_diff)})")

    print("### 上位三件:")
    # レポート対象週で活動があったプロジェクトのみを対象にソート
    sorted_projects_report = sorted(project_durations_report.items(), key=lambda item: item[1], reverse=True)
    for i, (project_name, duration) in enumerate(sorted_projects_report[:3]):
        percentage = (duration / total_duration_report * 100) if total_duration_report > 0 else 0
        comparison_duration = project_durations_comparison.get(project_name, 0)
        duration_diff = duration - comparison_duration
        print(f"{i+1}. {project_name}: {format_duration(duration)}({format_duration_diff(duration_diff)})  {percentage:.2f}%")
        if project_name == '雑務':
            # '雑務'プロジェクトのIDを逆引き
            zatsumu_project_id = next((pid for pid, name in project_id_to_name.items() if name == '雑務'), None)
            if zatsumu_project_id:
                zatsumu_tasks = [
                    entry['description'] for entry in filtered_time_entries_report_week
                    if entry.get('project_id') == zatsumu_project_id and entry.get('description')
                ]
                if zatsumu_tasks:
                    print(f"    内容: {', '.join(sorted(list(set(zatsumu_tasks))))}") # 重複を削除してソートして表示

    print("### 下位三件:")
    # レポート対象週で活動があったプロジェクトの中から、時間が短い順にソート
    active_projects_report_for_bottom = {name: duration for name, duration in project_durations_report.items() if duration > 0}
    # ゼロ時間のプロジェクトを除外しない場合は、上記フィルターを削除。
    # 例のように実際のタスクを表示するためにゼロ時間をフィルタリングします。

    sorted_projects_report_asc = sorted(active_projects_report_for_bottom.items(), key=lambda item: item[1])
    for i, (project_name, duration) in enumerate(sorted_projects_report_asc[:3]):
        percentage = (duration / total_duration_report * 100) if total_duration_report > 0 else 0
        comparison_duration = project_durations_comparison.get(project_name, 0)
        duration_diff = duration - comparison_duration
        print(f"{i+1}. {project_name}: {format_duration(duration)}({format_duration_diff(duration_diff)})  {percentage:.2f}%")
        if project_name == '雑務':
            # '雑務'プロジェクトのIDを逆引き
            zatsumu_project_id = next((pid for pid, name in project_id_to_name.items() if name == '雑務'), None)
            if zatsumu_project_id:
                zatsumu_tasks = [
                    entry['description'] for entry in filtered_time_entries_report_week
                    if entry.get('project_id') == zatsumu_project_id and entry.get('description')
                ]
                if zatsumu_tasks:
                    print(f"   * 内容: {', '.join(sorted(list(set(zatsumu_tasks))))}") # 重複を削除してソートして表示

# Get the content from the temporary buffer
captured_output = temp_buf.getvalue()

# Print the captured output to the cell's standard output
print(captured_output, file=sys.stdout)

# Append the captured output to the global list
_combined_report_sections.append(captured_output)

In [ ]:
import sys # <-- Add this import
import io
import contextlib

# Create a temporary buffer for current cell's output
temp_buf = io.StringIO()

with contextlib.redirect_stdout(temp_buf): # Redirect all prints within this block to temp_buf
    # --- クライアント別の作業時間集計と前週比 ---
    print("### クライアント別の作業時間")

    # プロジェクトIDからクライアントIDへのマッピングを作成
    project_id_to_client_id = {}
    for project in combined_workspace_data['projects']:
        if project.get('client_id'): # クライアントが紐付いているプロジェクトのみ対象
            project_id_to_client_id[project['id']] = project['client_id']

    # クライアントIDからクライアント名へのマッピングを作成
    client_id_to_name = {}
    for client in combined_workspace_data['clients']:
        client_id_to_name[client['id']] = client['name']

    # クライアント名からクライアントIDへの逆マッピングも作成しておく
    client_name_to_id = {v: k for k, v in client_id_to_name.items()}

    def process_time_entries_by_client(entries_list, project_id_to_client_id_map, client_id_to_name_map):
        client_durations = {}
        for entry in entries_list:
            # まず、エントリ自体にclient_idが含まれているかを確認 (CSVから読み込まれたデータに存在する可能性)
            client_id = entry.get('client_id')

            # エントリにclient_idがない場合、project_idからマッピングを試みる
            if client_id is None:
                project_id = entry.get('project_id')
                client_id = project_id_to_client_id_map.get(project_id)

            if client_id is not None: # クライアントに紐づくプロジェクトの場合
                client_name = client_id_to_name_map.get(client_id, 'Unknown Client')
                duration = entry.get('duration', 0)
                client_durations[client_name] = client_durations.get(client_name, 0) + duration
            else: # クライアントに紐づかないプロジェクト、またはプロジェクトなしの場合
                # 'No Client' として集計することも可能だが、今回はクライアントに紐づくもののみを対象とする
                pass

        return client_durations

    # レポート対象週のクライアント別作業時間
    client_durations_report = process_time_entries_by_client(
        filtered_time_entries_report_week, project_id_to_client_id, client_id_to_name
    )

    # 比較対象週のクライアント別作業時間
    client_durations_comparison = process_time_entries_by_client(
        filtered_time_entries_comparison_week, project_id_to_client_id, client_id_to_name
    )

    if client_durations_report:
        # 全クライアントの合計作業時間を計算
        total_client_duration_report = sum(client_durations_report.values())

        # 作業時間の多い順にソート
        sorted_client_durations = sorted(client_durations_report.items(), key=lambda item: item[1], reverse=True)

        for client_name, duration in sorted_client_durations:
            percentage = (duration / total_client_duration_report * 100) if total_client_duration_report > 0 else 0
            comparison_duration = client_durations_comparison.get(client_name, 0)
            duration_diff = duration - comparison_duration
            print(f"* {client_name}: {format_duration(duration)}({format_duration_diff(duration_diff)})  {percentage:.2f}%")

        # 個人ワーククライアントのプロジェクトリスト表示
        personal_work_client_id = client_name_to_id.get('個人ワーク')
        if personal_work_client_id:
            # レポート対象週で活動があったプロジェクトIDのセットを取得
            active_project_ids_in_report_week = {entry['project_id'] for entry in filtered_time_entries_report_week if 'project_id' in entry and entry['project_id'] is not None}

            personal_work_projects_with_activity = []
            for project in combined_workspace_data['projects']:
                project_id = project.get('id')
                # 個人ワーククライアントに属し、かつレポート対象週に活動があったプロジェクトのみをリストアップ
                if project.get('client_id') == personal_work_client_id and project_id in active_project_ids_in_report_week:
                    personal_work_projects_with_activity.append(project.get('name', 'Unknown Project'))

            if personal_work_projects_with_activity:
                # 重複を削除し、ソートして表示
                unique_sorted_projects = sorted(list(set(personal_work_projects_with_activity)))
                print(f"    * 個人ワークに割り当てられたプロジェクト: {', '.join(unique_sorted_projects)}")
            else:
                print("  個人ワークに割り当てられたプロジェクトはありませんでした。")


    else:
        print(" クライアントに紐づく作業時間がありませんでした。")

# Get the content from the temporary buffer
captured_output = temp_buf.getvalue()

# Print the captured output to the cell's standard output
print(captured_output, file=sys.stdout)

# Append the captured output to the global list
_combined_report_sections.append(captured_output)

In [ ]:
import locale
from datetime import timedelta
import sys # <-- Add this import

# 睡眠、趣味に関する詳細レポートを生成するヘルパー関数
def analyze_category_duration(category_name, report_entries, comparison_entries, project_id_map, jst_timezone, start_date_report_week_dt):
    category_id = None
    for p_id, p_name in project_id_map.items():
        if p_name == category_name:
            category_id = p_id
            break

    if category_id is None:
        print(f"Warning: カテゴリ '{category_name}' がレポート対象週のプロジェクトマッピングに見つかりませんでした。")
        return None # Category not found

    # Filter entries for the specific category
    category_report_entries = [e for e in report_entries if e.get('project_id') == category_id]
    category_comparison_entries = [e for e in comparison_entries if e.get('project_id') == category_id]

    # Calculate total duration for report and comparison weeks
    total_duration_report = sum(e.get('duration', 0) for e in category_report_entries)
    total_duration_comparison = sum(e.get('duration', 0) for e in category_comparison_entries)
    duration_diff = total_duration_report - total_duration_comparison

    # Add a specific warning if comparison data for this category is zero but report data is not.
    if total_duration_report > 0 and total_duration_comparison == 0:
        print(f"--- 警告: '{category_name}' の比較データ ---")
        print(f"  '{category_name}' の前週の比較データがゼロであるか、見つかりませんでした。")
        print("  これは、前週のCSVファイルが 'プロジェクトID' および 'クライアントID' 列を含むように生成される前に作成されたか、前週にそのプロジェクトが存在しなかった、または活動がなかったためである可能性があります。")
        print("  比較週のCSVファイル（例: 'toggl_activity_log_full_2026-08-23.csv'）にこれらのID列が含まれ、正しく読み込まれていることを確認してください。")
        print("------------------------------------\n")

    # Calculate daily durations for the report week
    daily_durations = {}
    # Initialize all 7 days of the week with 0 duration
    for i in range(7):
        date = (start_date_report_week_dt + timedelta(days=i)).date()
        daily_durations[date] = 0

    for entry in category_report_entries:
        entry_date = entry['start_jst'].date()
        daily_durations[entry_date] = daily_durations.get(entry_date, 0) + entry.get('duration', 0)

    longest_day_duration = 0
    longest_day_date = None
    second_longest_day_duration = 0
    second_longest_day_date = None
    shortest_day_duration = None # Initialize with None to handle cases where all days are 0 or no entries
    shortest_day_date = None

    if total_duration_report == 0: # All days are 0 for this category in the report week
        # If all durations are 0, then longest, second longest, and shortest are all 0
        longest_day_duration = 0
        longest_day_date = None # or any date if preferred, but None makes sense
        second_longest_day_duration = 0
        second_longest_day_date = None
        shortest_day_duration = 0
        shortest_day_date = None # or any date if preferred
    else:
        # Check for days with 0 duration first
        zero_duration_days = [date for date, duration in daily_durations.items() if duration == 0]
        if zero_duration_days: # If there are days with 0 duration, that's the shortest
            shortest_day_duration = 0
            shortest_day_date = sorted(zero_duration_days)[0] # Take the earliest date with 0 duration

            # For longest/second longest, only consider days with > 0 duration
            positive_daily_durations = {k: v for k, v in daily_durations.items() if v > 0}
            sorted_positive_durations = sorted(positive_daily_durations.items(), key=lambda item: item[1], reverse=True)
            if sorted_positive_durations:
                longest_day_date = sorted_positive_durations[0][0]
                longest_day_duration = sorted_positive_durations[0][1]
            if len(sorted_positive_durations) > 1:
                second_longest_day_date = sorted_positive_durations[1][0]
                second_longest_day_duration = sorted_positive_durations[1][1]
        else: # All days have positive duration, so find the min/max among them
            # Sort all daily durations (which are all > 0 here) to find longest and second longest
            sorted_daily_durations = sorted(daily_durations.items(), key=lambda item: item[1], reverse=True)

            if sorted_daily_durations:
                longest_day_date = sorted_daily_durations[0][0]
                longest_day_duration = sorted_daily_durations[0][1]

            if len(sorted_daily_durations) > 1:
                second_longest_day_date = sorted_daily_durations[1][0]
                second_longest_day_duration = sorted_daily_durations[1][1]

            shortest_day_duration = min(daily_durations.values())
            for date, duration in daily_durations.items():
                if duration == shortest_day_duration:
                    shortest_day_date = date
                    break # Take the first one if multiple days have the min duration

    # Calculate daily average for the report week
    daily_average_duration = total_duration_report / 7

    return {
        'total_duration_report': total_duration_report,
        'daily_average': daily_average_duration,
        'longest_day_duration': longest_day_duration,
        'longest_day_date': longest_day_date,
        'second_longest_day_duration': second_longest_day_duration,
        'second_longest_day_date': second_longest_day_date,
        'shortest_day_duration': shortest_day_duration,
        'shortest_day_date': shortest_day_date,
        'duration_diff': duration_diff
    }

# Create a temporary buffer for current cell's output
temp_buf = io.StringIO()

with contextlib.redirect_stdout(temp_buf): # Redirect all prints within this block to temp_buf
    # Add new sections for '睡眠' and '趣味'

    # --- 睡眠 (Sleep) ---
    sleep_data = analyze_category_duration(
        '睡眠',
        time_entries_report_week,
        time_entries_comparison_week,
        project_id_to_name,
        jst,
        start_time_report_week_jst
    )

    if sleep_data:
        print("\n## 睡眠💤")
        print(f"* 週合計: {format_duration(sleep_data['total_duration_report'])}({format_duration_diff(sleep_data['duration_diff'])})")
        print(f"* 日平均: {format_duration(sleep_data['daily_average'])}(前週との日平均差分: {format_duration_diff(sleep_data['duration_diff'] / 7 if sleep_data['duration_diff'] is not None else None)}) ")
        if sleep_data['longest_day_date']:
            print(f"* 最も長かった日: {sleep_data['longest_day_date'].strftime('%Y/%m/%d(%a)')} - {format_duration(sleep_data['longest_day_duration'])}")
        else:
            print("* 最も長かった日: データなし")
        if sleep_data['shortest_day_date'] is not None:
            print(f"* 最も短かった日: {sleep_data['shortest_day_date'].strftime('%Y/%m/%d(%a)')} - {format_duration(sleep_data['shortest_day_duration'])}")
        else:
            print("* 最も短かった日: データなし")
    else:
        print("\n## 睡眠💤")
        print(" '睡眠' カテゴリのデータが見つかりませんでした。")

    # --- 趣味 (Hobby) ---
    hobby_data = analyze_category_duration(
        '趣味',
        time_entries_report_week,
        time_entries_comparison_week,
        project_id_to_name,
        jst,
        start_time_report_week_jst
    )

    if hobby_data:
        print("\n## 趣味🎮")
        print(f"* 週合計: {format_duration(hobby_data['total_duration_report'])}({format_duration_diff(hobby_data['duration_diff'])})")
        print(f"* 日平均: {format_duration(hobby_data['daily_average'])}(前週との日平均差分: {format_duration_diff(hobby_data['duration_diff'] / 7 if hobby_data['duration_diff'] is not None else None)}) ")
        if hobby_data['longest_day_date']:
            print(f"* 最も長かった日: {hobby_data['longest_day_date'].strftime('%Y/%m/%d(%a)')} - {format_duration(hobby_data['longest_day_duration'])}")
        else:
            print("* 最も長かった日: データなし")
        if hobby_data['second_longest_day_date']:
            print(f"* 二番目に長かった日: {hobby_data['second_longest_day_date'].strftime('%Y/%m/%d(%a)')} - {format_duration(hobby_data['second_longest_day_duration'])}")
        else:
            print("* 二番目に長かった日: データなし")
        if hobby_data['shortest_day_date'] is not None:
            print(f"* 最も短かった日: {hobby_data['shortest_day_date'].strftime('%Y/%m/%d(%a)')} - {format_duration(hobby_data['shortest_day_duration'])}")
        else:
            print("* 最も短かった日: データなし")
    else:
        print("\n## 趣味🎮")
        print(" '趣味' カテゴリのデータが見つかりませんでした。")

# Get the content from the temporary buffer
captured_output = temp_buf.getvalue()

# Print the captured output to the cell's standard output
print(captured_output, file=sys.stdout)

# Append the captured output to the global list
_combined_report_sections.append(captured_output)


In [ ]:
import io
import contextlib
import sys # <-- Add this import
from datetime import timedelta, datetime

def get_daily_category_durations(category_name, time_entries, project_id_map, start_date_report_week_dt):
    category_id = None
    for p_id, p_name in project_id_map.items():
        if p_name == category_name:
            category_id = p_id
            break

    daily_durations = {}
    # Initialize all 7 days of the week with 0 duration
    for i in range(7):
        date = (start_date_report_week_dt + timedelta(days=i)).date()
        daily_durations[date] = 0

    if category_id is not None:
        for entry in time_entries:
            if entry.get('project_id') == category_id:
                entry_date = entry['start_jst'].date()
                daily_durations[entry_date] = daily_durations.get(entry_date, 0) + entry.get('duration', 0)
    return daily_durations

# Create a temporary buffer for current cell's output
temp_buf = io.StringIO()

with contextlib.redirect_stdout(temp_buf): # Redirect all prints within this block to temp_buf
    print("\n## 睡眠と趣味のグラフ📊")

    # Check if required variables are defined. If not, print an error and skip chart generation.
    required_vars = ['time_entries_report_week', 'project_id_to_name', 'start_time_report_week_jst']
    missing_vars = [var for var in required_vars if var not in globals()]

    if missing_vars:
        print(f"グラフの生成に必要なデータの一部が見つかりません。不足している変数: {', '.join(missing_vars)}。前のセルをすべて実行してください。")
    else:
        # Get daily durations for Sleep and Hobby
        sleep_daily = get_daily_category_durations('睡眠', time_entries_report_week, project_id_to_name, start_time_report_week_jst)
        hobby_daily = get_daily_category_durations('趣味', time_entries_report_week, project_id_to_name, start_time_report_week_jst)

        # Prepare data for Mermaid chart
        chart_dates = sorted(sleep_daily.keys()) # Dates will be the same for both
        sleep_hours = [round(sleep_daily[date] / 3600, 2) for date in chart_dates]
        hobby_hours = [round(hobby_daily[date] / 3600, 2) for date in chart_dates]
        date_labels = [date.strftime('%d') for date in chart_dates]

        # Determine max value for y-axis to ensure adequate scaling
        max_y_value = 0
        if sleep_hours: max_y_value = max(max_y_value, max(sleep_hours))
        if hobby_hours: max_y_value = max(max_y_value, max(hobby_hours))
        y_axis_max = max(1, round(max_y_value * 1.2)) # Add 20% buffer, ensure at least 1

        # Generate Mermaid xychart
        mermaid_chart = '''```mermaid
---
config:
  themeVariables:
    xyChart:
      plotColorPalette: '#0000FF, #FF00FF'
---
xychart-beta
    title "睡眠と趣味 (時間)"
    x-axis "日付" [{}]
    y-axis "時間 (h)" 0 --> {}

    line "睡眠" [{}]
    line "趣味" [{}]
```'''.format(
            ", ".join(f'"{label}"' for label in date_labels),
            y_axis_max,
            ", ".join(map(str, sleep_hours)),
            ", ".join(map(str, hobby_hours))
        )
        print(mermaid_chart)

# Get the content from the temporary buffer
captured_output = temp_buf.getvalue()

# Print the captured output to the cell's standard output
print(captured_output, file=sys.stdout) # <-- Explicitly print to sys.stdout

# Append the captured output to the global list
_combined_report_sections.append(captured_output)


In [ ]:
# Display the combined report
for section in _combined_report_sections:
    print(section)

### 週次活動レポートの生成

計算された総作業時間、日平均、上位・下位のプロジェクト（タスク）リスト、および前週との差分を、指定されたフォーマットで出力します。

### 前提データの準備

`NameError: name 'processed_time_entries' is not defined`というエラーが発生しました。これは、タイムエントリデータ(`processed_time_entries`)や、それに付随するプロジェクト、クライアント情報、日付範囲などの変数が、このセルを実行する前に定義されていないために発生します。

以下のセルでは、必要なデータをToggl Track APIから取得し、レポート生成に必要な形式に変換するための変数やヘルパー関数を定義します。これにより、後続のCSV出力セルが正しく動作するようになります。

In [ ]:
import requests

# Toggl Track APIからタイムエントリデータを取得
# time_entries_url, headers, auth, params は前のセルで定義済みと仮定
try:
    response = requests.get(time_entries_url, headers=headers, auth=auth, params=params)
    response.raise_for_status()  # HTTPエラー (4xx または 5xx) が発生した場合に例外を発生させる
    time_entries_data = response.json()
    print(f"APIから {len(time_entries_data)} 件のタイムエントリを正常に取得しました。")
except requests.exceptions.RequestException as e:
    print(f"データの取得中にエラーが発生しました: {e}")
    if 'response' in locals() and response.status_code == 403:
        print("APIトークンを確認し、正しいことを確認してください。")
    time_entries_data = [] # エラーが発生した場合はデータを空にする

APIから 186 件のタイムエントリを正常に取得しました。


In [ ]:
from google.colab import userdata

# ワークスペースIDとラベルを定義
# TOGGL_WORK_WORKSPACE_ID, TOGGL_LIFE_WORKSPACE_ID はColab secretsから取得済みと仮定
work_workspace_id = userdata.get('TOGGL_WORK_WORKSPACE_ID')
life_workspace_id = userdata.get('TOGGL_LIFE_WORKSPACE_ID')

if not work_workspace_id or not life_workspace_id:
    raise ValueError("TogglワークスペースIDが見つかりません。'TOGGL_WORK_WORKSPACE_ID'と'TOGGL_LIFE_WORKSPACE_ID'をColab secretsに追加してください。")

workspaces_to_fetch = {
    'work': int(work_workspace_id),
    'life': int(life_workspace_id)
}

combined_workspace_data = {
    'projects': [],
    'clients': []
}

print("指定されたワークスペースのプロジェクトおよびクライアントデータを取得中...")

for label, workspace_id in workspaces_to_fetch.items():
    try:
        # 現在のワークスペースのプロジェクトデータを取得
        all_projects_for_workspace = []

        # アクティブなプロジェクトを取得
        projects_url_active = f"{TOGGL_API_BASE_URL}/workspaces/{workspace_id}/projects?active=true"
        projects_response_active = requests.get(projects_url_active, headers=headers, auth=auth)
        projects_response_active.raise_for_status()
        active_projects_data = projects_response_active.json()
        all_projects_for_workspace.extend(active_projects_data)

        # アーカイブされたプロジェクトを取得
        projects_url_archived = f"{TOGGL_API_BASE_URL}/workspaces/{workspace_id}/projects?active=false"
        projects_response_archived = requests.get(projects_url_archived, headers=headers, auth=auth)
        projects_response_archived.raise_for_status()
        archived_projects_data = projects_response_archived.json()
        all_projects_for_workspace.extend(archived_projects_data)

        # ワークスペースラベルを各プロジェクトに追加し、結合リストに追加
        for project in all_projects_for_workspace:
            project['workspace_label'] = label
            combined_workspace_data['projects'].append(project)

        # 現在のワークスペースのクライアントデータを取得
        clients_url = f"{TOGGL_API_BASE_URL}/workspaces/{workspace_id}/clients"
        clients_response = requests.get(clients_url, headers=headers, auth=auth)
        clients_response.raise_for_status()
        clients_data = clients_response.json()

        # ワークスペースラベルを各クライアントに追加し、結合リストに追加
        for client in clients_data:
            client['workspace_label'] = label
            combined_workspace_data['clients'].append(client)

    except requests.exceptions.RequestException as e:
        print(f"ワークスペース {label} (ID: {workspace_id}) のデータ取得中にエラーが発生しました: {e}")
        if 'projects_response_active' in locals() and hasattr(projects_response_active, 'status_code') and projects_response_active.status_code == 403:
            print(f"ワークスペース {workspace_id} へのAPIトークンのアクセスを確認してください。")
        elif 'clients_response' in locals() and hasattr(clients_response, 'status_code') and clients_response.status_code == 403:
            print(f"ワークスペース {workspace_id} へのAPIトークンのアクセスを確認してください。")

print("ワークスペース、プロジェクト、クライアントデータの取得が完了しました。")

指定されたワークスペースのプロジェクトおよびクライアントデータを取得中...
ワークスペース、プロジェクト、クライアントデータの取得が完了しました。


In [ ]:
import pytz
from datetime import datetime, timedelta

jst = pytz.timezone('Asia/Tokyo')
now_utc = datetime.now(pytz.utc)
now_jst = now_utc.astimezone(jst)

# --- レポート対象週の期間計算 (先週の月曜日 - 直前の日曜日) ---
# 今日の曜日 (0=月, 6=日) に基づいて、直前の日曜日を特定
days_ago_to_last_sunday = (now_jst.weekday() + 1) % 7
last_sunday_date_00_00_jst_naive = now_jst.date() - timedelta(days=days_ago_to_last_sunday)
last_sunday_date_00_00_jst = jst.localize(datetime(last_sunday_date_00_00_jst_naive.year,
                                                    last_sunday_date_00_00_jst_naive.month,
                                                    last_sunday_date_00_00_jst_naive.day,
                                                    0, 0, 0))

# レポート対象週の終了日 (直前の日曜日 23:59:59 JST)
end_time_report_week_jst = last_sunday_date_00_00_jst + timedelta(days=1) - timedelta(seconds=1)
# レポート対象週の開始日 (その6日前の月曜日 00:00:00 JST)
start_time_report_week_jst = last_sunday_date_00_00_jst - timedelta(days=6)

# レポートファイル名の基準日をレポート対象週の終了日とする
report_date_jst = end_time_report_week_jst.date()

# time_entries_dataが空でないことを確認
if not time_entries_data:
    print("APIから取得したタイムエントリが見つからないため、レポート対象週データは空になります。")
    time_entries_report_week = [] # APIデータがない場合、空にする
else:
    # タイムエントリの'start'と'stop'をJST datetimeオブジェクトに変換
    processed_time_entries_temp = [] # Use a temporary variable for processing
    for entry in time_entries_data:
        entry_copy = entry.copy()
        if 'start' in entry_copy and entry_copy['start']:
            start_utc = datetime.fromisoformat(entry_copy['start'].replace('Z', '+00:00'))
            entry_copy['start_jst'] = start_utc.astimezone(jst)
        else:
            entry_copy['start_jst'] = None
        if 'stop' in entry_copy and entry_copy['stop']:
            stop_utc = datetime.fromisoformat(entry_copy['stop'].replace('Z', '+00:00'))
            entry_copy['stop_jst'] = stop_utc.astimezone(jst)
        else:
            entry_copy['stop_jst'] = None
        processed_time_entries_temp.append(entry_copy)

    # レポート対象週のタイムエントリをフィルタリング
    time_entries_report_week = [
        entry for entry in processed_time_entries_temp
        if entry['start_jst'] and start_time_report_week_jst <= entry['start_jst'] <= end_time_report_week_jst
    ]
    print(f"レポート対象週のタイムエントリを処理しました: {len(time_entries_report_week)} 件。")

レポート対象週のタイムエントリを処理しました: 186 件。


In [ ]:
# Toggl TrackのプロジェクトIDとプロジェクト名をマッピングするための辞書を作成
# combined_workspace_data は前のセルで取得済みと仮定
project_id_to_name = {}
project_id_to_client_id = {}
for project in combined_workspace_data['projects']:
    project_id_to_name[project['id']] = project['name']
    if project.get('client_id'): # クライアントが紐付いている場合のみ追加
        project_id_to_client_id[project['id']] = project['client_id']

# クライアントIDからクライアント名へのマッピングを作成
client_id_to_name = {}
for client in combined_workspace_data['clients']:
    client_id_to_name[client['id']] = client['name']

# ワークスペースIDからラベルへのマッピングを作成
# workspaces_to_fetch は前のセルで定義済みと仮定
workspace_id_to_label = {v: k for k, v in workspaces_to_fetch.items()}

# 時間をH:MM:SS形式にフォーマットするヘルパー関数
def format_duration(seconds):
    if seconds is None:
        return "N/A"
    seconds = int(seconds)
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    return f"{hours:02}:{minutes:02}:{secs:02}"

print("プロジェクト、クライアントマッピング、およびヘルパー関数が定義されました。")

プロジェクト、クライアントマッピング、およびヘルパー関数が定義されました。


In [ ]:
import pandas as pd
from google.colab import files

# ワークスペースIDからラベルへのマッピングを作成
workspace_id_to_label = {v: k for k, v in workspaces_to_fetch.items()}

csv_rows = []
for entry in processed_time_entries:
    # ワークスペース名
    workspace_name = workspace_id_to_label.get(entry.get('workspace_id'), 'Unknown Workspace')

    # プロジェクト名
    project_name = project_id_to_name.get(entry.get('project_id'), 'No Project')

    # タスク名 (descriptionを使用)
    task_name = entry.get('description', '')

    # 開始日時と終了日時 (JSTでフォーマット)
    start_time_formatted = entry['start_jst'].strftime('%Y-%m-%d %H:%M:%S') if entry['start_jst'] else ''
    stop_time_formatted = entry['stop_jst'].strftime('%Y-%m-%d %H:%M:%S') if entry['stop_jst'] else ''

    # 経過時間 (format_duration関数を使用)
    duration_formatted = format_duration(entry.get('duration'))

    # タグ (コロン区切り)
    tags = entry.get('tags', [])
    tags_str = ":".join(tags) if tags else ''

    csv_rows.append({
        'ワークスペース名': workspace_name,
        'プロジェクト名': project_name,
        'タスク名': task_name,
        '開始日時': start_time_formatted,
        '終了日時': stop_time_formatted,
        '経過時間': duration_formatted,
        'タグ': tags_str
    })

# DataFrameを作成
df_activity_log = pd.DataFrame(csv_rows)

# CSVファイルとして保存
# report_date_jst は前方のセルで定義済みと仮定
output_filename = f"toggl_activity_log_full_{report_date_jst.strftime('%Y-%m-%d')}.csv"
df_activity_log.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"行動ログが '{output_filename}' としてCSV形式で保存されました。")
display(df_activity_log.head())
files.download(output_filename)


NameError: name 'processed_time_entries' is not defined

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Define the path to save the CSV in Google Drive
gdrive_output_path = f'/content/drive/MyDrive/TogglTrack_Reports/{output_filename}'

# Save the DataFrame to Google Drive
df_activity_log.to_csv(gdrive_output_path, index=False, encoding='utf-8-sig')
print(f"行動ログが '{gdrive_output_path}' としてGoogle Driveに保存されました。")

NameError: name 'output_filename' is not defined

# デバッグ用

In [ ]:
print("\n--- 'No Project' に分類されたタスクの詳細 (レポート対象週) ---")

no_project_entries = []
for entry in filtered_time_entries_report_week:
    project_id = entry.get('project_id')
    # プロジェクトIDがNoneの場合、またはproject_id_to_nameに存在しない場合
    if project_id is None or project_id not in project_id_to_name:
        no_project_entries.append(entry)

if no_project_entries:
    # 説明と継続時間を表示
    for entry in no_project_entries:
        description = entry.get('description', '説明なし')
        duration = entry.get('duration', 0)
        print(f"Description: {description}, Duration: {format_duration(duration)}")
else:
    print("レポート対象週に 'No Project' に分類されたタスクはありませんでした。")